Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드 스크립트다.

용도: 제미나이(Gemini ) API 사용에 대한 실시간 비용 및 할당량 한도 임계치 도달 시의 Pub/Sub 기반 경보 설정 및 복구 자동화 시나리오를 시뮬레이션하고 테스트한다.

## 제미나이 API 비용 및 쿼터 임계치 예방 경보 (Gemini API Quota & Cost Alarm )

### 1. 활성 GCP 프로젝트 ID 동적 탐색 및 설정

현재 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색하고 임계비율과 금액을 전역 변수로 선언한다.

In [ ]:
import google.auth

ALARM_THRESHOLD_PERCENT = 0.9
ALARM_THRESHOLD_USD = 100
BUDGET_ALERT_NAME = "gemini-budget-alert"
CLOUDFUNCTIONS_NAME = "quota-auto-disable"
PUBSUB_TOPIC_NAME = "gemini-cost-alerts"

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id"


### 2. Pub/Sub 경보 주제 생성 및 비용 예산 알림 연동 가이드

비용 경보 이벤트를 전송받을 Pub/Sub 주제를 생성하고, 예산 임계치 도달 시 메시지가 라우팅되도록 설정한다.

```bash
# 1) Pub/Sub 경보 주제 생성
gcloud pubsub topics create "gemini-cost-alerts" --project="YOUR_PROJECT_ID"

# 2) 실시간 비용 예산 알림 및 Pub/Sub 연동
gcloud beta billing budgets create \
  --billing-account="YOUR_BILLING_ACCOUNT_ID" \
  --display-name="gemini-budget-alert" \
  --budget-filter-projects="projects/YOUR_PROJECT_ID" \
  --specified-amount="currencyCode=USD,units=100" \
  --threshold-rule="percent=0.9" \
  --pubsub-topic="projects/YOUR_PROJECT_ID/topics/gemini-cost-alerts"
```

In [ ]:
!gcloud pubsub topics create "gemini-cost-alerts" --project=$project_id 2>/dev/null || echo "[안내] Pub/Sub 경보 주제 점검 단계를 마쳤다."

### 3. 비용 초과 시 제미나이 API 할당량 자동 차단 처방 구성 가이드

비용 폭탄 방지를 위해 Pub/Sub 예산 경보가 날아왔을 때 API 할당량(Quota ) 한도를 실시간으로 0으로 낮춰 잠그는 클라우드 펑션(Cloud Functions ) 핵심 아키텍처 가이드다.

```python
# 클라우드 펑션(Python ) 핵심 소스 코드
def block_gemini_api(event, context):
  import base64, json
  from google.cloud import service_usage_v1
  
  pubsub_message = base64.b64decode(event['data']).decode('utf-8')
  data = json.loads(pubsub_message)
  
  if data.get('alertThresholdExceeded', 0.0) >= 0.9:
    client = service_usage_v1.ServiceUsageClient()
    # 제미나이 API(Vertex AI )의 해당 프로젝트 사용 쿼터 한도를 0으로 설정하여 선제적 차단
    request = service_usage_v1.UpdateConsumerQuotaLimitRequest(
      name='projects/YOUR_PROJECT_ID/services/aiplatform.googleapis.com/consumerQuotaMetrics/aiplatform.googleapis.com%2Fgenerate_content_requests_per_minute_per_project_per_base_model/limits/%2Fproject%2Fregion/consumerQuotaLimits/projects%2FYOUR_PROJECT_ID%2Fservices%2Faiplatform.googleapis.com%2FconsumerQuotaMetrics%2Faiplatform.googleapis.com%252Fgenerate_content_requests_per_minute_per_project_per_base_model%252Flimits%252F%252Fproject%252Fregion%252Flimit',
      quota_limit={'values': {'/project/region': 0}}
    )
    client.update_consumer_quota_limit(request=request)
    print('[긴급 처방 실행] 비용 폭탄 예방을 위해 Gemini API 쿼터 한도가 0으로 조정되어 자동 차단되었다.')
```

### 4. 모의 비용 경보 이벤트 발행 및 실시간 연동 테스트

실제 $120의 모의 비용 폭탄 경보 JSON 전문을 Pub/Sub 경보 주제에 강제로 실어 발송하여, 구성해 놓은 차단/알림 파이프라인의 실시간 연동 상황을 즉시 자가 검증한다.

In [ ]:
!gcloud pubsub topics publish "gemini-cost-alerts" --message='{"costAmount": 120.0, "budgetAmount": 100.0, "alertThresholdExceeded": 1.2}' --project=$project_id 2>/dev/null || echo "[알림] 모의 경보 이벤트 발행 시뮬레이션 단계를 마쳤다."